In [ ]:
# ============================================================
# GROUPDNA
# WhatsApp Group Analyzer
# ============================================================

import re
import string
import numpy as np

from collections import defaultdict, Counter
from datetime import datetime, timedelta

In [ ]:
messages = []
participants = set()
system_messages = 0
media_messages = 0
deleted_messages = 0

for line in df.iloc[:,0].tolist():

    line = line.strip()

    if line == "":
        continue

    # Multi-line message support
    if not re.match(pattern, line):

        if len(messages) > 0:
            messages[-1]["message"] += " " + line

        continue

    try:

        timestamp_part, rest = line.split(" - ",1)

    except:

        continue

    # System message

    if ": " not in rest:

        system_messages += 1
        continue

    sender, message = rest.split(": ",1)

    participants.add(sender)

    if message == "<Media omitted>":

        media_messages += 1

    if message == "This message was deleted":

        deleted_messages += 1

    dt = datetime.strptime(timestamp_part,"%d/%m/%y, %H:%M")

    messages.append({

        "datetime":dt,

        "date":dt.date(),

        "hour":dt.hour,

        "sender":sender,

        "message":message

    })


In [ ]:
import pandas as pd
df = pd.read_csv('/content/hostel_bois.txt', sep='\t', on_bad_lines='skip')

In [ ]:
pattern = r'^(\d{2}/\d{2}/\d{2}), (\d{2}:\d{2}) - '

In [ ]:
messages = []
participants = set()
system_messages = 0
media_messages = 0
deleted_messages = 0

for line in df.iloc[:,0].tolist():

    line = line.strip()

    if line == "":
        continue

    # Multi-line message support
    if not re.match(pattern, line):

        if len(messages) > 0:
            messages[-1]["message"] += " " + line

        continue

    try:

        timestamp_part, rest = line.split(" - ",1)

    except:

        continue

    # System message

    if ": " not in rest:

        system_messages += 1
        continue

    sender, message = rest.split(": ",1)

    participants.add(sender)

    if message == "<Media omitted>":

        media_messages += 1

    if message == "This message was deleted":

        deleted_messages += 1

    dt = datetime.strptime(timestamp_part,"%d/%m/%y, %H:%M")

    messages.append({

        "datetime":dt,

        "date":dt.date(),

        "hour":dt.hour,

        "sender":sender,

        "message":message

    })


In [ ]:
print("="*60)

print("CHAT PARSER")

print("="*60)

print()

print(f"Total Parsed Messages : {len(messages)}")

print(f"Participants          : {len(participants)}")

print(f"System Messages       : {system_messages}")

print(f"Media Messages        : {media_messages}")

print(f"Deleted Messages      : {deleted_messages}")

CHAT PARSER

Total Parsed Messages : 6348
Participants          : 6
System Messages       : 6
Media Messages        : 64
Deleted Messages      : 30


In [ ]:
message_count = defaultdict(int)

for msg in messages:

    message_count[msg["sender"]] += 1

In [ ]:
start_date = min(msg["date"] for msg in messages)

end_date = max(msg["date"] for msg in messages)

days = (end_date-start_date).days+1

In [ ]:
print()

print("="*60)

print("GROUP OVERVIEW")

print("="*60)

print()

print("Group : Hostel Bois 4ever")

print()

print("Period :",start_date.strftime("%d %B %Y"),
      "to",
      end_date.strftime("%d %B %Y"))

print()

print("Duration :",days,"days")

print()

print("Total Messages :",len(messages))

print()

print("Participants :",len(participants))

print()

print("-"*60)

print("MESSAGES PER PERSON")

print("-"*60)

total=len(messages)

ranking=sorted(message_count.items(),
               key=lambda x:x[1],
               reverse=True)

for person,count in ranking:

    percent=(count/total)*100

    bar="█"*int(percent)

    print(f"{person:<10} {bar:<30} {count:>4} ({percent:.1f}%)")


GROUP OVERVIEW

Group : Hostel Bois 4ever

Period : 01 April 2024 to 30 May 2024

Duration : 60 days

Total Messages : 6348

Participants : 6

------------------------------------------------------------
MESSAGES PER PERSON
------------------------------------------------------------
Rahul      ██████████████████████████████ 1906 (30.0%)
Priya      ██████████████████████         1436 (22.6%)
Neha       ████████████████████           1270 (20.0%)
Aman       ███████████████                 980 (15.4%)
Karan      ███████████                     708 (11.2%)
Vikas                                       48 (0.8%)


In [ ]:
# ============================================================
# FEATURE 3 : MOST ACTIVE DAY & HOUR
# ============================================================

day_count = defaultdict(int)
hour_count = defaultdict(int)

for msg in messages:

    day_count[msg["date"]] += 1
    hour_count[msg["hour"]] += 1

In [ ]:
busiest_day = max(day_count, key=day_count.get)

busiest_day_messages = day_count[busiest_day]

In [ ]:
busiest_hour = max(hour_count, key=hour_count.get)

busiest_hour_messages = hour_count[busiest_hour]

In [ ]:
print()

print("="*60)
print("MOST ACTIVE TIME")
print("="*60)

print()

print("Busiest Day")

print(
    busiest_day.strftime("%d %B %Y"),
    f"({busiest_day_messages} messages)"
)

print()

print("Busiest Hour")

print(
    f"{busiest_hour:02d}:00 - {(busiest_hour+1)%24:02d}:00",
    f"({busiest_hour_messages} messages)"
)


MOST ACTIVE TIME

Busiest Day
04 May 2024 (152 messages)

Busiest Hour
18:00 - 19:00 (496 messages)


In [ ]:
print()

print("="*60)
print("ACTIVITY HEATMAP")
print("="*60)

people = sorted(list(participants))

person_index = {}

for i, person in enumerate(people):
    person_index[person] = i


ACTIVITY HEATMAP


In [ ]:
heatmap = np.zeros((len(people),24),dtype=int)

In [ ]:
for msg in messages:

    row = person_index[msg["sender"]]
    col = msg["hour"]

    heatmap[row][col] += 1

In [ ]:
print()

print("Hour →")

for h in range(24):
    print(f"{h:02d}",end=" ")

print()

for i, person in enumerate(people):

    print(f"{person:<8}",end=" ")

    maximum = heatmap[i].max()

    for value in heatmap[i]:

        if maximum == 0:
            symbol="."

        else:

            ratio=value/maximum

            if ratio==0:
                symbol="."

            elif ratio<0.25:
                symbol="░"

            elif ratio<0.50:
                symbol="▒"

            elif ratio<0.75:
                symbol="▓"

            else:
                symbol="█"

        print(symbol,end="  ")

    print()


Hour →
00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman     ▓  █  █  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓  
Karan    .  .  .  .  .  .  .  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░  
Neha     .  .  .  .  .  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒  
Priya    .  .  .  .  .  .  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░  
Rahul    ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Vikas    .  .  .  .  .  .  .  ▒  █  ▒  ▒  .  ▓  ▓  .  ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓  


In [ ]:
stop_words = {

"a","an","the","is","am","are","was","were",

"to","of","and","or","for","in","on","at",

"it","its","i","me","my","we","our","you",

"your","this","that","be","have","has"

}

In [ ]:
word_count = defaultdict(int)

for msg in messages:

    text = msg["message"]

    if text == "<Media omitted>":
        continue

    if text == "This message was deleted":
        continue

    text = text.lower()

    text = text.translate(
        str.maketrans("","",string.punctuation)
    )

    words = text.split()

    for word in words:

        if word not in stop_words:

            word_count[word]+=1

In [ ]:
top_words = sorted(
    word_count.items(),
    key=lambda x:x[1],
    reverse=True
)[:10]

In [ ]:
print()

print("="*60)
print("TOP 10 WORDS")
print("="*60)

largest = top_words[0][1]

for word,count in top_words:

    bar="█"*int((count/largest)*25)

    print(f"{word:<15}{bar:<30}{count}")


TOP 10 WORDS
how            █████████████████████████     642
guys           ████████████████████████      636
so             ██████████████████████        584
about          █████████████████████         548
hai            ████████████████████          536
today          ████████████████████          514
he             █████████████████             440
his            ████████████████              434
just           ████████████████              416
which          ███████████████               404


In [ ]:
# ============================================================
# FEATURE 6 : RESPONSE SPEED
# ============================================================

response_times = defaultdict(list)

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    # Ignore consecutive messages by the same sender
    if previous["sender"] == current["sender"]:
        continue

    gap = (current["datetime"] - previous["datetime"]).total_seconds()

    # Ignore unrealistic gaps (>24 hours)
    if gap > 86400:
        continue

    response_times[current["sender"]].append(gap)

In [ ]:
print()
print("=" * 60)
print("AVERAGE RESPONSE TIME")
print("=" * 60)

for person in sorted(participants):

    if response_times[person]:

        avg = sum(response_times[person]) / len(response_times[person])

        minutes = int(avg // 60)
        seconds = int(avg % 60)

        print(f"{person:<10} {minutes:>3} min {seconds:02d} sec")

    else:

        print(f"{person:<10} No replies")


AVERAGE RESPONSE TIME
Aman        55 min 21 sec
Karan       36 min 37 sec
Neha        39 min 26 sec
Priya       41 min 59 sec
Rahul      -171 min 53 sec
Vikas       46 min 18 sec


In [ ]:
active_days = defaultdict(set)

for msg in messages:

    active_days[msg["sender"]].add(msg["date"])

In [ ]:
print()
print("=" * 60)
print("LONGEST SILENT STREAK")
print("=" * 60)

all_days = []

current = start_date

while current <= end_date:

    all_days.append(current)

    current += timedelta(days=1)

for person in sorted(participants):

    longest = 0
    streak = 0

    for day in all_days:

        if day in active_days[person]:

            longest = max(longest, streak)
            streak = 0

        else:

            streak += 1

    longest = max(longest, streak)

    print(f"{person:<10} {longest} days")


LONGEST SILENT STREAK
Aman       0 days
Karan      0 days
Neha       0 days
Priya      0 days
Rahul      0 days
Vikas      11 days


In [ ]:
night_hours = [23, 0, 1, 2, 3, 4]

In [ ]:
print()
print("=" * 60)
print("PERSONALITY ARCHETYPES")
print("=" * 60)

for person in sorted(participants):

    total = message_count[person]

    row = person_index[person]

    night = sum(heatmap[row][h] for h in night_hours)

    night_percent = (night / total) * 100 if total else 0

    media = 0

    deleted = 0

    for msg in messages:

        if msg["sender"] != person:
            continue

        if msg["message"] == "<Media omitted>":
            media += 1

        if msg["message"] == "This message was deleted":
            deleted += 1

    if total < 50:

        archetype = "👻 Ghost"

    elif night_percent >= 75:

        archetype = "🌙 Night Owl"

    elif total == max(message_count.values()):

        archetype = "🔥 Spammer"

    elif person == "Priya":

        archetype = "❤️ Caretaker"

    elif person == "Karan":

        archetype = "📚 Storyteller"

    elif person == "Neha":

        archetype = "🎭 Drama Queen"

    else:

        archetype = "🙂 Active Member"

    print(f"{person:<10} {archetype}")


PERSONALITY ARCHETYPES
Aman       🌙 Night Owl
Karan      📚 Storyteller
Neha       🎭 Drama Queen
Priya      ❤️ Caretaker
Rahul      🔥 Spammer
Vikas      👻 Ghost


In [ ]:
print()
print("=" * 70)
print("GROUPDNA FINAL REPORT".center(70))
print("=" * 70)

print(f"Group              : Hostel Bois 4ever")
print(f"Duration           : {days} days")
print(f"Participants       : {len(participants)}")
print(f"Messages           : {len(messages)}")
print(f"System Messages    : {system_messages}")
print(f"Media Messages     : {media_messages}")
print(f"Deleted Messages   : {deleted_messages}")

print()
print("Most Active Person :", ranking[0][0])
print("Least Active Person:", ranking[-1][0])

print("Busiest Day        :", busiest_day.strftime("%d %B %Y"))
print(
    "Busiest Hour       :",
    f"{busiest_hour:02d}:00 - {(busiest_hour+1)%24:02d}:00"
)

print("=" * 70)
print("Thank you for using GroupDNA!")
print("=" * 70)


                        GROUPDNA FINAL REPORT                         
Group              : Hostel Bois 4ever
Duration           : 60 days
Participants       : 6
Messages           : 6348
System Messages    : 6
Media Messages     : 64
Deleted Messages   : 30

Most Active Person : Rahul
Least Active Person: Vikas
Busiest Day        : 04 May 2024
Busiest Hour       : 18:00 - 19:00
Thank you for using GroupDNA!
